# 07 – Statistical Analysis

This notebook performs a Welch independent-samples t-test comparing review scores for delayed and on-time delivered orders, and a one-way ANOVA comparing order values across product categories. Statistical association does not establish causation.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/Cart2Insights")
CLEAN = ROOT / "data" / "cleaned"
RAW = ROOT / "data" / "raw"

if not CLEAN.exists():
    raise FileNotFoundError("Project data/cleaned folder was not found.")

print("Cleaned data folder:", CLEAN)


In [ ]:

from scipy import stats

orders = pd.read_csv(CLEAN / "orders_cleaned.csv")
reviews = pd.read_csv(CLEAN / "reviews_cleaned.csv")

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"], errors="coerce"
)
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"], errors="coerce"
)

orders["is_delayed"] = (
    orders["order_delivered_customer_date"] >
    orders["order_estimated_delivery_date"]
)

review_data = orders[["order_id", "is_delayed"]].merge(
    reviews[["order_id", "review_score"]], on="order_id", how="inner"
).dropna()

delayed = review_data.loc[
    review_data["is_delayed"], "review_score"
]
on_time = review_data.loc[
    ~review_data["is_delayed"], "review_score"
]

print("Delayed reviews:", len(delayed))
print("On-time reviews:", len(on_time))

if len(delayed) > 1 and len(on_time) > 1:
    result = stats.ttest_ind(
        delayed, on_time, equal_var=False, nan_policy="omit"
    )
    print("Welch t-test statistic:", result.statistic)
    print("p-value:", result.pvalue)
else:
    print("Not enough observations for the t-test.")


In [ ]:

# One-way ANOVA: order value across product categories
items = pd.read_csv(CLEAN / "order_items_cleaned.csv")
products = pd.read_csv(CLEAN / "products_cleaned.csv")
translation = pd.read_csv(CLEAN / "category_translation_cleaned.csv")

category_data = items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id", how="left"
).merge(
    translation, on="product_category_name", how="left"
)

category_data["category"] = category_data[
    "product_category_name_english"
].fillna(category_data["product_category_name"])

category_data["order_value"] = (
    category_data["price"] + category_data["freight_value"]
)

groups = [
    group["order_value"].dropna().values
    for _, group in category_data.groupby("category")
    if len(group["order_value"].dropna()) > 1
]

if len(groups) >= 2:
    result = stats.f_oneway(*groups)
    print("ANOVA F-statistic:", result.statistic)
    print("p-value:", result.pvalue)
else:
    print("Not enough category groups for ANOVA.")
